# Loss Functions

A loss function measures **how wrong** your model is.
The lower the loss, the better the model. Training = minimizing the loss.

Choosing the right loss function depends on your task.

In [ ]:
import torch
import torch.nn as nn

## 1. MSE Loss — Mean Squared Error (Regression)

**Math:** L = (1/n) Σ (y_pred - y_true)²

**Use when:** predicting continuous values (house prices, temperatures, stock values)

In [ ]:
mse = nn.MSELoss()

y_pred = torch.tensor([2.5, 3.0, 4.0])
y_true = torch.tensor([3.0, 3.0, 5.0])

loss = mse(y_pred, y_true)
print('MSE loss:', loss.item())
# Manual: ((2.5-3)^2 + (3-3)^2 + (4-5)^2) / 3 = (0.25 + 0 + 1) / 3 = 0.4167
print('Manual:  ', ((y_pred - y_true)**2).mean().item())

## 2. Cross Entropy Loss (Multi-class Classification)

**Math:** L = -Σ y_true * log(softmax(y_pred))

**Use when:** predicting one class out of N (image classification, token prediction in LLMs)

**Important:** PyTorch's CrossEntropyLoss expects raw logits (NOT softmaxed). It applies softmax internally.

In [ ]:
ce_loss = nn.CrossEntropyLoss()

# 3 examples, 4 classes
# logits — raw scores BEFORE softmax
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.5],   # example 1
    [0.5, 2.5, 0.3, 1.0],   # example 2
    [0.1, 0.2, 3.0, 0.5],   # example 3
])

# targets — class indices (NOT one-hot vectors)
targets = torch.tensor([0, 1, 2])   # correct class for each example

loss = ce_loss(logits, targets)
print('CrossEntropy loss:', loss.item())

# What CrossEntropyLoss does internally:
probs = torch.softmax(logits, dim=1)
print('\nProbabilities (softmax):')
print(probs)
print('Model picked:', probs.argmax(dim=1))   # predicted class
print('Correct:     ', targets)

## 3. BCE Loss — Binary Cross Entropy (Binary Classification)

**Math:** L = -(y * log(p) + (1-y) * log(1-p))

**Use when:** yes/no prediction (spam/not spam, fraud/not fraud)

In [ ]:
# BCEWithLogitsLoss = Sigmoid + BCE in one (numerically stable, preferred)
bce_loss = nn.BCEWithLogitsLoss()

logits  = torch.tensor([2.0, -1.0, 0.5, -2.0])   # raw scores
targets = torch.tensor([1.0,  0.0, 1.0,  0.0])   # 1 = positive, 0 = negative

loss = bce_loss(logits, targets)
print('BCE loss:', loss.item())

# Predicted probabilities (for display)
probs = torch.sigmoid(logits)
print('Predicted probs:', probs)
print('Predicted class:', (probs > 0.5).int())   # threshold at 0.5
print('Actual:         ', targets.int())

## Loss function cheat sheet

| Task | Output layer | Loss function |
|---|---|---|
| Regression | nn.Linear (no activation) | nn.MSELoss |
| Binary classification | nn.Linear (no activation) | nn.BCEWithLogitsLoss |
| Multi-class classification | nn.Linear (no activation) | nn.CrossEntropyLoss |
| Multi-label classification | nn.Linear (no activation) | nn.BCEWithLogitsLoss |
| Language model (next token) | nn.Linear (vocab size) | nn.CrossEntropyLoss |

Notice: **never apply softmax/sigmoid yourself** when using PyTorch loss functions — they do it internally for numerical stability.

In [ ]:
# What loss looks like at the start of training (random weights) vs end (trained)
torch.manual_seed(42)

model    = nn.Linear(4, 3)   # 3-class classifier
loss_fn  = nn.CrossEntropyLoss()
x        = torch.randn(8, 4)
targets  = torch.randint(0, 3, (8,))

logits = model(x)
loss   = loss_fn(logits, targets)
print(f'Loss with random weights: {loss.item():.4f}')
print(f'Expected random loss:     {torch.log(torch.tensor(3.0)).item():.4f}')  # -log(1/3)
# For N classes, random loss ≈ log(N) — a useful sanity check